# Notebook 19 — Stale Context

**Continual Learning in Context: RML Extension for CL-BENCH**

Notebook 17 examined plasticity:

> Can the system incorporate useful new structure after context changes?

Notebook 19 asks the failure-side question:

> When does retained context stop helping and start becoming noise?

In continual learning, memory is useful only while the underlying context remains valid.

Stale context appears when prior structure is reused after it no longer transfers.

Outputs:

- `results/19_stale_context_events.csv`
- `results/19_stale_risk_by_variant.csv`
- `results/19_stale_context_summary.json`
- `figures/19_stale_risk_timeline.png`
- `figures/19_stale_risk_by_variant.png`
- `figures/19_gain_drop_events.png`
- `figures/19_stale_vs_plasticity.png`
- `results/19_stale_context_artifacts.zip`

## 0. Bootstrap Repo Path

This cell works locally or in Colab.

If opened directly in Colab and the repo is missing, it clones:

```text
https://github.com/thinkthoughts/continual-learning-bench-rml
```

In [ ]:
from pathlib import Path
import sys
import os
import subprocess

REPO_URL = "https://github.com/thinkthoughts/continual-learning-bench-rml.git"
REPO_NAME = "continual-learning-bench-rml"

def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

def find_rml_root(start: Path) -> Path | None:
    start = start.resolve()
    for base in [start, *start.parents]:
        if (base / "src" / "gain.py").exists():
            return base
        if (base / "rml" / "src" / "gain.py").exists():
            return base / "rml"
        if (base / REPO_NAME / "rml" / "src" / "gain.py").exists():
            return base / REPO_NAME / "rml"
    return None

cwd = Path.cwd().resolve()
RML_ROOT = find_rml_root(cwd)

if RML_ROOT is None and running_in_colab():
    repo_path = Path("/content") / REPO_NAME

    if repo_path.exists():
        print("Repo already exists. Pulling latest changes...")
        subprocess.run(["git", "-C", str(repo_path), "pull"], check=False)
    else:
        print("Cloning repo...")
        subprocess.run(["git", "clone", REPO_URL, str(repo_path)], check=True)

    RML_ROOT = repo_path / "rml"
    os.chdir(RML_ROOT)

elif RML_ROOT is not None:
    os.chdir(RML_ROOT)

else:
    raise RuntimeError(
        "Could not locate rml/src/gain.py. "
        "Run this notebook inside the repo, or in Colab allow the bootstrap cell to clone the repo."
    )

if str(RML_ROOT) not in sys.path:
    sys.path.insert(0, str(RML_ROOT))

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Running in Colab:", running_in_colab())
print("Current working directory:", Path.cwd())
print("RML root:", RML_ROOT)
print("results:", RESULTS_DIR)
print("figures:", FIGURES_DIR)

## 1. Imports

In [ ]:
import json
import zipfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.gain import compute_gain
from src.drift import detect_drift

print("Imports complete.")

## 2. Load Prior Artifacts

Notebook 19 prefers the plasticity-state output from Notebook 17:

```text
results/17_plasticity_states.csv
```

It also uses optional Notebook 17 recovery data when available.

Fallback order for trajectory:

1. `results/17_plasticity_states.csv`
2. `results/13_context_stability.csv`
3. `results/11_state_vs_stateless_instances.csv`
4. `results/01_gain_signal_trajectory.csv`
5. `results/00_context_gain.csv`
6. synthetic toy trajectory

In [ ]:
trajectory_17 = RESULTS_DIR / "17_plasticity_states.csv"
trajectory_13 = RESULTS_DIR / "13_context_stability.csv"
trajectory_11 = RESULTS_DIR / "11_state_vs_stateless_instances.csv"
trajectory_01 = RESULTS_DIR / "01_gain_signal_trajectory.csv"
trajectory_00 = RESULTS_DIR / "00_context_gain.csv"

if trajectory_17.exists():
    df = pd.read_csv(trajectory_17)
    source_file = trajectory_17
    print("Loaded:", trajectory_17)
elif trajectory_13.exists():
    df = pd.read_csv(trajectory_13)
    source_file = trajectory_13
    print("Loaded:", trajectory_13)
elif trajectory_11.exists():
    df = pd.read_csv(trajectory_11)
    source_file = trajectory_11
    print("Loaded:", trajectory_11)
elif trajectory_01.exists():
    df = pd.read_csv(trajectory_01)
    source_file = trajectory_01
    print("Loaded:", trajectory_01)
elif trajectory_00.exists():
    df = pd.read_csv(trajectory_00)
    source_file = trajectory_00
    print("Loaded:", trajectory_00)
else:
    print("No prior trajectory found. Creating fallback toy trajectory.")
    df = pd.DataFrame({
        "instance": np.arange(1, 13),
        "variant": [
            "A", "A", "A", "A",
            "B", "B", "B", "B",
            "C", "C", "C", "C",
        ],
        "stateful_reward": [
            0.18, 0.22, 0.28, 0.35,
            0.43, 0.48, 0.46, 0.52,
            0.40, 0.45, 0.55, 0.60,
        ],
        "stateless_reward": [
            0.18, 0.19, 0.20, 0.21,
            0.22, 0.20, 0.21, 0.22,
            0.23, 0.22, 0.24, 0.25,
        ],
    })
    source_file = None

if "gain" not in df.columns:
    if "state_advantage" in df.columns:
        df["gain"] = df["state_advantage"]
    else:
        df["gain"] = compute_gain(
            df["stateful_reward"].tolist(),
            df["stateless_reward"].tolist(),
        )

if "gain_delta" not in df.columns:
    df["gain_delta"] = df["gain"].diff().fillna(0.0)

if "cumulative_gain" not in df.columns:
    df["cumulative_gain"] = df["gain"].cumsum()

df = df.sort_values("instance").reset_index(drop=True)

recovery_path = RESULTS_DIR / "17_boundary_recovery.csv"
if recovery_path.exists():
    boundary_recovery = pd.read_csv(recovery_path)
    print("Loaded:", recovery_path)
else:
    boundary_recovery = None
    print("No boundary recovery file found.")

df

## 3. Define Stale Context

Stale context is not simply low performance.

Stale context means:

> retained state threatens usefulness because the current context has changed.

Signals used here:

- negative gain
- large negative gain delta
- boundary transfer cost
- declining after prior high gain
- low recovery after a transition

Notebook 19 does not claim these are definitive failures. It provides a **risk score** for stale-context candidates.

## 4. Variant Boundaries

In [ ]:
variant_order = (
    df[["variant", "instance"]]
    .drop_duplicates("variant")
    .sort_values("instance")
)

boundary_instances = variant_order["instance"].tolist()

boundary_table = pd.DataFrame({
    "variant": variant_order["variant"].tolist(),
    "boundary_instance": boundary_instances,
})

boundary_table

## 5. Stale-Context Event Detection

This creates several interpretable flags:

- `negative_gain_flag`
- `large_gain_drop_flag`
- `boundary_flag`
- `decline_after_high_gain_flag`
- `low_recovery_flag`

In [ ]:
events = df.copy()

negative_threshold = -0.02
drop_threshold = -0.08

events["negative_gain_flag"] = events["gain"] < negative_threshold
events["large_gain_drop_flag"] = events["gain_delta"] <= drop_threshold
events["boundary_flag"] = events["instance"].isin(boundary_instances)

# Decline after previously high gain.
high_gain_threshold = float(events["gain"].quantile(0.75))
events["prior_gain"] = events["gain"].shift(1).fillna(0.0)

events["decline_after_high_gain_flag"] = (
    (events["prior_gain"] >= high_gain_threshold)
    & (events["gain_delta"] < 0)
)

# Low recovery flag from Notebook 17 boundary recovery, when available.
events["low_recovery_flag"] = False

if boundary_recovery is not None:
    recovery_lookup = {
        row["variant"]: row["recovery_gain"]
        for _, row in boundary_recovery.iterrows()
    }
    median_recovery = float(boundary_recovery["recovery_gain"].median())

    for i, row in events.iterrows():
        variant = row["variant"]
        if row["boundary_flag"] and variant in recovery_lookup:
            events.loc[i, "low_recovery_flag"] = recovery_lookup[variant] < median_recovery

events[[
    "instance",
    "variant",
    "gain",
    "gain_delta",
    "negative_gain_flag",
    "large_gain_drop_flag",
    "boundary_flag",
    "decline_after_high_gain_flag",
    "low_recovery_flag",
]]

## 6. Stale Risk Score

The stale risk score aggregates the event flags.

The score is intentionally simple:

\[
\text{risk}
=
1(negative)
+
1(large\ drop)
+
0.5(boundary)
+
1(decline\ after\ high\ gain)
+
1(low\ recovery)
\]

Higher values mean stronger stale-context risk.

In [ ]:
events["stale_risk_score"] = (
    events["negative_gain_flag"].astype(float)
    + events["large_gain_drop_flag"].astype(float)
    + 0.5 * events["boundary_flag"].astype(float)
    + events["decline_after_high_gain_flag"].astype(float)
    + events["low_recovery_flag"].astype(float)
)

def classify_risk(score):
    if score >= 2.0:
        return "high"
    if score >= 1.0:
        return "medium"
    if score > 0:
        return "low"
    return "none"

events["stale_risk_level"] = events["stale_risk_score"].apply(classify_risk)

events[[
    "instance",
    "variant",
    "gain",
    "gain_delta",
    "stale_risk_score",
    "stale_risk_level",
]]

## 7. Stale Risk by Variant

In [ ]:
risk_by_variant = (
    events.groupby("variant")
    .agg(
        instances=("instance", "count"),
        mean_gain=("gain", "mean"),
        min_gain=("gain", "min"),
        mean_gain_delta=("gain_delta", "mean"),
        total_stale_risk=("stale_risk_score", "sum"),
        mean_stale_risk=("stale_risk_score", "mean"),
        high_risk_instances=("stale_risk_level", lambda x: int((x == "high").sum())),
        medium_risk_instances=("stale_risk_level", lambda x: int((x == "medium").sum())),
        low_risk_instances=("stale_risk_level", lambda x: int((x == "low").sum())),
        no_risk_instances=("stale_risk_level", lambda x: int((x == "none").sum())),
    )
    .reset_index()
)

risk_by_variant

## 8. Figure — Stale Risk Timeline

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    events["instance"],
    events["stale_risk_score"],
)

for boundary in boundary_instances[1:]:
    ax.axvline(boundary - 0.5, linestyle=":", linewidth=1)

ax.set_title("Notebook 19: Stale Context Risk Timeline")
ax.set_xlabel("Instance")
ax.set_ylabel("Stale Risk Score")
ax.grid(True, axis="y", alpha=0.3)

stale_risk_timeline_path = FIGURES_DIR / "19_stale_risk_timeline.png"
fig.tight_layout()
fig.savefig(stale_risk_timeline_path, dpi=160)

stale_risk_timeline_path

## 9. Figure — Stale Risk by Variant

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(
    risk_by_variant["variant"],
    risk_by_variant["mean_stale_risk"],
)

ax.axhline(0, linewidth=1)
ax.set_title("Notebook 19: Mean Stale Risk by Variant")
ax.set_xlabel("Variant")
ax.set_ylabel("Mean Stale Risk Score")
ax.grid(True, axis="y", alpha=0.3)

stale_risk_variant_path = FIGURES_DIR / "19_stale_risk_by_variant.png"
fig.tight_layout()
fig.savefig(stale_risk_variant_path, dpi=160)

stale_risk_variant_path

## 10. Figure — Gain Drop Events

Large negative gain deltas are one of the clearest signs of stale-context risk.

In [ ]:
drop_events = events.copy()
drop_events["drop_magnitude"] = drop_events["gain_delta"].apply(
    lambda x: abs(x) if x < 0 else 0.0
)

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    drop_events["instance"],
    drop_events["drop_magnitude"],
)

for boundary in boundary_instances[1:]:
    ax.axvline(boundary - 0.5, linestyle=":", linewidth=1)

ax.set_title("Notebook 19: Gain Drop Events")
ax.set_xlabel("Instance")
ax.set_ylabel("Negative Gain Delta Magnitude")
ax.grid(True, axis="y", alpha=0.3)

gain_drop_events_path = FIGURES_DIR / "19_gain_drop_events.png"
fig.tight_layout()
fig.savefig(gain_drop_events_path, dpi=160)

gain_drop_events_path

## 11. Figure — Stale Risk vs Plasticity

This figure compares stale risk to gain adaptation.

High risk with positive recovery means the system had to revise but recovered.

High risk without recovery would be a stronger stale-memory failure.

In [ ]:
if boundary_recovery is not None:
    stale_vs_plasticity = risk_by_variant.merge(
        boundary_recovery[["variant", "recovery_gain"]],
        on="variant",
        how="left",
    )
else:
    # Fallback: estimate recovery as max gain minus first gain within variant.
    rows = []
    for variant, group in events.groupby("variant", sort=False):
        group = group.sort_values("instance")
        rows.append({
            "variant": variant,
            "recovery_gain": float(group["gain"].max() - group["gain"].iloc[0]),
        })
    recovery_fallback = pd.DataFrame(rows)
    stale_vs_plasticity = risk_by_variant.merge(
        recovery_fallback,
        on="variant",
        how="left",
    )

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(
    stale_vs_plasticity["mean_stale_risk"],
    stale_vs_plasticity["recovery_gain"],
    s=120,
)

for _, row in stale_vs_plasticity.iterrows():
    ax.annotate(
        row["variant"],
        (row["mean_stale_risk"], row["recovery_gain"]),
        textcoords="offset points",
        xytext=(8, 8),
    )

ax.axhline(0, linewidth=1)
ax.set_title("Notebook 19: Stale Risk vs Plasticity")
ax.set_xlabel("Mean Stale Risk")
ax.set_ylabel("Recovery Gain")
ax.grid(True, alpha=0.3)

stale_vs_plasticity_path = FIGURES_DIR / "19_stale_vs_plasticity.png"
fig.tight_layout()
fig.savefig(stale_vs_plasticity_path, dpi=160)

stale_vs_plasticity

## 12. Summary

In [ ]:
highest_risk_row = events.loc[events["stale_risk_score"].idxmax()].to_dict()
highest_variant_row = risk_by_variant.loc[
    risk_by_variant["mean_stale_risk"].idxmax()
].to_dict()

summary = {
    "total_instances": int(len(events)),
    "variants": events["variant"].drop_duplicates().tolist(),
    "trajectory_source": str(source_file) if source_file is not None else "fallback",
    "negative_gain_instances": int(events["negative_gain_flag"].sum()),
    "large_gain_drop_instances": int(events["large_gain_drop_flag"].sum()),
    "decline_after_high_gain_instances": int(events["decline_after_high_gain_flag"].sum()),
    "low_recovery_boundary_instances": int(events["low_recovery_flag"].sum()),
    "total_stale_risk": float(events["stale_risk_score"].sum()),
    "mean_stale_risk": float(events["stale_risk_score"].mean()),
    "highest_risk_instance": int(highest_risk_row["instance"]),
    "highest_risk_variant": str(highest_risk_row["variant"]),
    "highest_risk_score": float(highest_risk_row["stale_risk_score"]),
    "highest_risk_variant_by_mean": str(highest_variant_row["variant"]),
    "highest_mean_variant_risk": float(highest_variant_row["mean_stale_risk"]),
}

summary

## 13. Export Artifacts

In [ ]:
events_csv = RESULTS_DIR / "19_stale_context_events.csv"
risk_by_variant_csv = RESULTS_DIR / "19_stale_risk_by_variant.csv"
stale_vs_plasticity_csv = RESULTS_DIR / "19_stale_vs_plasticity.csv"
summary_path = RESULTS_DIR / "19_stale_context_summary.json"
zip_path = RESULTS_DIR / "19_stale_context_artifacts.zip"

events.to_csv(events_csv, index=False)
risk_by_variant.to_csv(risk_by_variant_csv, index=False)
stale_vs_plasticity.to_csv(stale_vs_plasticity_csv, index=False)

summary_with_metadata = {
    **summary,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "19_stale_context.ipynb",
    "extension": "continual-learning-bench-rml",
    "repo": REPO_URL,
}

with open(summary_path, "w") as f:
    json.dump(summary_with_metadata, f, indent=2)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in [
        events_csv,
        risk_by_variant_csv,
        stale_vs_plasticity_csv,
        summary_path,
        stale_risk_timeline_path,
        stale_risk_variant_path,
        gain_drop_events_path,
        stale_vs_plasticity_path,
    ]:
        z.write(path, arcname=path.name)

print("Saved artifacts:")
for path in [
    events_csv,
    risk_by_variant_csv,
    stale_vs_plasticity_csv,
    summary_path,
    stale_risk_timeline_path,
    stale_risk_variant_path,
    gain_drop_events_path,
    stale_vs_plasticity_path,
    zip_path,
]:
    print("-", path)

## 14. Download Artifacts

In Colab, this downloads the zip.

Locally, the archive is saved under:

```text
rml/results/19_stale_context_artifacts.zip
```

In [ ]:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Download helper is only active in Colab.")
    print("Artifact archive:", zip_path)

## 15. Notebook 19 Claim

Memory helps only while context remains valid.

When context changes, retained structure can become stale.

Continual learning therefore requires more than retention:

\[
\text{learn}
\rightarrow
\text{retain}
\rightarrow
\text{revise}
\rightarrow
\text{discard when stale}
\]

In RML terms:

> stale context is structure that once helped but now constrains adaptation.

Notebook 23 will next examine drift adaptation:

> Can systems revise stale assumptions when the environment changes?